# EXAONE 3-way 평가: fine-tuned vs base vs GPT-4o-mini

파인튜닝된 EXAONE(HF Hub 어댑터), base EXAONE(파인튜닝 X), GPT-4o-mini 세 모델을 동일한 test set(201건)으로 평가.

**설계 원칙**
- Gemini는 baseline에서 제외 (gold label 자체가 Gemini로 만들어져서 순환 논리가 됨)
- 세 모델 모두 SYSTEM_PROMPT(파인튜닝 학습 때 쓴 것과 동일)를 그대로 사용 — base/GPT가 emoji-text 포맷을 못 지키는 것 자체가 `parse_fail_rate`라는 유의미한 지표
- 생성(generation)과 채점(scoring)을 분리해서 각 모델 결과를 먼저 jsonl로 저장 → 세션 끊겨도 재현 가능

## 0. 패키지 설치 + Drive 마운트

In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub openai scikit-learn scipy tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 설정값

In [6]:
from google.colab import userdata

LABELED_PATH = "/content/drive/MyDrive/Colab Notebooks/AIM/AIHub_면접데이터/labeled_dataset.jsonl"        # 원본 라벨링 데이터 (gold 구조화 라벨용)
SAVE_DIR = "/content/drive/MyDrive/exaone_qlora_v2"     # v2 평가 결과 저장 위치
TEST_SET_PATH = "/content/drive/MyDrive/Colab Notebooks/AIM/exaone_qlora_v1/test_set.jsonl"   # v1 test split 그대로 재사용 (question/answer만 씀, 포맷 무관하게 재사용 가능)

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
ADAPTER_REPO_ID = "shk776/exaone-interview-adapter-v2"

HF_TOKEN = userdata.get('HF_TOKEN')                   # HF 토큰 (read 권한)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')       # OpenAI API 키

# v2 학습 때 쓴 SYSTEM_PROMPT와 정확히 동일해야 함 (fine-tuned 모델이 학습 분포 안에서 평가받도록)
SYSTEM_PROMPT = (
    "너는 대기업 전문 채용 면접관이자 AI 취업 코치이다.\n"
    "제시된 [면접 질문]과 [사용자 답변]을 분석하여 아래 4가지 항목을 평가하라.\n\n"
    "1. 두괄식: 답변이 결론부터 시작하는지 true/false로 평가하라.\n"
    "2. 논리구조: 답변이 '결론 -> 근거 -> 마무리'의 논리적 흐름을 갖추고 있는지 1~5점으로 평가하고, 그 이유를 설명하라.\n"
    "3. 키워드: 답변에서 발견된 키워드와 누락된 키워드를 각각 나열하라.\n"
    "4. 분량: 답변 글자수를 세고, 150~400자 기준으로 적정한지 평가하라.\n\n"
    "반드시 아래의 JSON 포맷으로만 응답하며, JSON 외의 서론이나 설명은 절대 추가하지 마라.\n"
    "{\n"
    '  "headline": true 또는 false,\n'
    '  "logic_structure": 1~5 사이 정수,\n'
    '  "logic_reason": "논리구조 평가 이유",\n'
    '  "found_keywords": ["발견된 키워드"],\n'
    '  "missing_keywords": ["누락된 키워드"],\n'
    '  "char_count": 정수,\n'
    '  "is_appropriate": true 또는 false\n'
    "}"
)

## 2. Test set + gold 구조화 라벨 로딩
test_set.jsonl에는 텍스트 피드백만 있고 구조화 라벨(headline bool, logic 점수 등)은 없어서, 원본 labeled_dataset에서 (질문, 답변)으로 다시 매칭해 gold를 붙임.

In [8]:
import json, re

with open(TEST_SET_PATH, "r", encoding="utf-8") as f:
    raw_test_data = [json.loads(l) for l in f]

def extract_qa(entry):
    if "question" in entry and "answer" in entry:
        return entry["question"], entry["answer"]
    # v2 형식: question/answer가 messages 안 user 텍스트에 섞여있어서 다시 파싱
    user_content = next(m["content"] for m in entry["messages"] if m["role"] == "user")
    m = re.search(r"\[면접 질문\]:\s*(.*?)\n\[사용자 답변\]:\s*(.*)", user_content, re.DOTALL)
    return m.group(1).strip(), m.group(2).strip()

test_data = []
for entry in raw_test_data:
    q, a = extract_qa(entry)
    test_data.append({"question": q, "answer": a})

with open(LABELED_PATH, "r", encoding="utf-8") as f:
    labeled_rows = [json.loads(l) for l in f]

gold_lookup = {(r["question"], r["answer"]): r["labels"] for r in labeled_rows}

for ex in test_data:
    ex["gold"] = gold_lookup.get((ex["question"], ex["answer"]))

missing = sum(1 for ex in test_data if ex["gold"] is None)
print(f"test set: {len(test_data)}건 / gold 매칭 실패: {missing}건")

test set: 201건 / gold 매칭 실패: 0건


## 3. 파싱 함수 (JSON 파싱)
v2는 JSON으로 응답하도록 학습됨. 앞뒤에 잡담이 섞여도 첫 `{`~마지막 `}` 구간만 추출해서 `json.loads()`로 파싱. gold와 필드명이 동일해서 별도 매핑 불필요.

In [9]:
import json

REQUIRED_KEYS = ["headline", "logic_structure", "found_keywords", "missing_keywords", "char_count", "is_appropriate"]

def parse_feedback_text(text: str):
    """v2 JSON 응답 파싱. 실패하면 None."""
    try:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start == -1 or end == 0:
            return None
        obj = json.loads(text[start:end])
    except (json.JSONDecodeError, ValueError):
        return None

    if not all(k in obj for k in REQUIRED_KEYS):
        return None

    def to_bool(v):
        return v if isinstance(v, bool) else str(v).strip().lower() == "true"

    try:
        return {
            "headline": to_bool(obj["headline"]),
            "logic_structure": int(obj["logic_structure"]),
            "found_keywords": list(obj.get("found_keywords", [])),
            "missing_keywords": list(obj.get("missing_keywords", [])),
            "char_count": int(obj["char_count"]),
            "is_appropriate": to_bool(obj["is_appropriate"]),
        }
    except (ValueError, TypeError):
        return None

## 4. 공용 생성 함수

In [10]:
import torch
from tqdm import tqdm

def generate_with_model(model, tokenizer, question, answer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"[면접 질문]: {question}\n[사용자 답변]: {answer}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=450, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def run_and_save(model, tokenizer, out_filename):
    results = []
    for ex in tqdm(test_data):
        raw = generate_with_model(model, tokenizer, ex["question"], ex["answer"])
        results.append({
            "question": ex["question"], "answer": ex["answer"],
            "gold": ex["gold"], "raw_output": raw,
        })
    out_path = f"{SAVE_DIR}/{out_filename}"
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"저장 완료 -> {out_path}")

## 5. Fine-tuned EXAONE (HF Hub 어댑터 로드 후 생성)

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_for_ft = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
ft_model = PeftModel.from_pretrained(base_for_ft, ADAPTER_REPO_ID, token=HF_TOKEN)
ft_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO_ID, token=HF_TOKEN)
ft_model.eval()

run_and_save(ft_model, ft_tokenizer, "eval_finetuned.jsonl")

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.56GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.91M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

100%|██████████| 201/201 [40:18<00:00, 12.03s/it]

저장 완료 -> /content/drive/MyDrive/exaone_qlora_v2/eval_finetuned.jsonl


## 6. base EXAONE (어댑터 없이) — 메모리 정리 후 새로 로드

In [12]:
import gc

del ft_model, base_for_ft
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model.eval()

run_and_save(base_model, base_tokenizer, "eval_base.jsonl")

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/70.3k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/6.70k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.91M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

100%|██████████| 201/201 [36:04<00:00, 10.77s/it]

저장 완료 -> /content/drive/MyDrive/exaone_qlora_v2/eval_base.jsonl


## 7. GPT-4o-mini (API, GPU 불필요 — 위 셀들과 독립적으로 아무때나 실행 가능)

In [13]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

results = []
for ex in tqdm(test_data):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"[면접 질문]: {ex['question']}\n[사용자 답변]: {ex['answer']}"},
        ],
        temperature=0.3,
    )
    raw = resp.choices[0].message.content
    results.append({
        "question": ex["question"], "answer": ex["answer"],
        "gold": ex["gold"], "raw_output": raw,
    })

out_path = f"{SAVE_DIR}/eval_gpt4omini.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"저장 완료 -> {out_path}")

100%|██████████| 201/201 [05:46<00:00,  1.72s/it]

저장 완료 -> /content/drive/MyDrive/exaone_qlora_v2/eval_gpt4omini.jsonl


## 8. 3개 결과 취합 + 지표 계산

In [14]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

def score_model(path):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(l) for l in f]

    parsed_gold, parsed_pred, fail_count = [], [], 0
    for r in rows:
        p = parse_feedback_text(r["raw_output"])
        if p is None or r["gold"] is None:
            fail_count += 1
            continue
        parsed_gold.append(r["gold"])
        parsed_pred.append(p)

    n = len(rows)
    if not parsed_pred:
        return {"n": n, "parse_fail_rate": 1.0}

    parse_fail_rate = fail_count / n

    headline_acc = sum(g["headline"] == p["headline"] for g, p in zip(parsed_gold, parsed_pred)) / len(parsed_pred)
    length_acc = sum(g["is_appropriate"] == p["is_appropriate"] for g, p in zip(parsed_gold, parsed_pred)) / len(parsed_pred)

    gold_scores = [g["logic_structure"] for g in parsed_gold]
    pred_scores = [p["logic_structure"] for p in parsed_pred]
    spearman = spearmanr(gold_scores, pred_scores).correlation
    kappa = cohen_kappa_score(gold_scores, pred_scores, weights="quadratic")
    mae = sum(abs(g - p) for g, p in zip(gold_scores, pred_scores)) / len(gold_scores)

    tp = fp = fn = 0
    for g, p in zip(parsed_gold, parsed_pred):
        gold_set, pred_set = set(g["missing_keywords"]), set(p["missing_keywords"])
        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {
        "n": n, "parse_fail_rate": round(parse_fail_rate, 3),
        "headline_acc": round(headline_acc, 3), "length_acc": round(length_acc, 3),
        "logic_spearman": round(spearman, 3) if spearman == spearman else None,
        "logic_kappa": round(kappa, 3), "logic_mae": round(mae, 3),
        "keyword_f1": round(f1, 3),
    }

summary = {}
for name, path in [
    ("fine-tuned", f"{SAVE_DIR}/eval_finetuned.jsonl"),
    ("base", f"{SAVE_DIR}/eval_base.jsonl"),
    ("gpt-4o-mini", f"{SAVE_DIR}/eval_gpt4omini.jsonl"),
]:
    summary[name] = score_model(path)
    print(name, summary[name])

fine-tuned {'n': 201, 'parse_fail_rate': 0.0, 'headline_acc': 0.721, 'length_acc': 0.886, 'logic_spearman': np.float64(0.339), 'logic_kappa': 0.387, 'logic_mae': 0.493, 'keyword_f1': 0.277}
base {'n': 201, 'parse_fail_rate': 0.02, 'headline_acc': 0.447, 'length_acc': 0.746, 'logic_spearman': np.float64(0.191), 'logic_kappa': 0.014, 'logic_mae': 1.467, 'keyword_f1': 0.018}
gpt-4o-mini {'n': 201, 'parse_fail_rate': 0.0, 'headline_acc': 0.692, 'length_acc': 0.811, 'logic_spearman': np.float64(0.432), 'logic_kappa': 0.31, 'logic_mae': 0.597, 'keyword_f1': 0.037}


## 9. 비교표 저장 (보고서용)

In [15]:
import pandas as pd

df = pd.DataFrame(summary).T
print(df)

df.to_csv(f"{SAVE_DIR}/3way_comparison_summary.csv", encoding="utf-8-sig")
print(f"\n저장 완료 -> {SAVE_DIR}/3way_comparison_summary.csv")

                 n  parse_fail_rate  headline_acc  length_acc  logic_spearman  \
fine-tuned   201.0             0.00         0.721       0.886           0.339   
base         201.0             0.02         0.447       0.746           0.191   
gpt-4o-mini  201.0             0.00         0.692       0.811           0.432   

             logic_kappa  logic_mae  keyword_f1  
fine-tuned         0.387      0.493       0.277  
base               0.014      1.467       0.018  
gpt-4o-mini        0.310      0.597       0.037  

저장 완료 -> /content/drive/MyDrive/exaone_qlora_v2/3way_comparison_summary.csv
